In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding, SimpleRNN, Dense

In [7]:

sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels= [1]*15 + [0]*15
labels= np.array(labels)

In [8]:
vocab_size= 2000
tok= Tokenizer(num_words= vocab_size,oov_token= " ")
tok.fit_on_texts(sentences)
seq= tok.texts_to_sequences(sentences)
maxlen= max(len(s)for s in seq)
X= pad_sequences(seq,maxlen= maxlen,padding= "post")
y= labels

In [9]:
X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [10]:
embed_dim= 16
rnn_units= 8

In [11]:
inp= Input(shape= (maxlen,), dtype="int32",name="inp")
x= Embedding(input_dim=vocab_size,output_dim=embed_dim,mask_zero=True,name='embed')(inp)
rnn= SimpleRNN(units=rnn_units,return_sequences= False,return_state=False,name='simple_rnn')
x_last= rnn(x)
out= Dense(1,activation= 'sigmoid',name='out')(x_last)
model= Model(inputs=inp,outputs=out)
model.compile(optimizer= 'adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp (InputLayer)    │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ inp[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ inp[0][0]         │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.fit(X,y, epochs=25, batch_size=8, verbose=1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5000 - loss: 0.6968
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6000 - loss: 0.6812
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6667 - loss: 0.6673
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7333 - loss: 0.6540
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8333 - loss: 0.6398
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8333 - loss: 0.6247
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9000 - loss: 0.6100
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9000 - loss: 0.5939
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9000 - loss: 0.5756
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9000 - loss: 0.5581
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9333 - loss: 0.5382
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9333 - loss: 0.5176
E

To inspect the intermediate hidden states, we can define an intermediate model that outputs the activation of the `SimpleRNN` layer. This model will take the same input as our main model.

In [13]:
from tensorflow.keras.models import Model
intermediate_model = Model(inputs=model.input, outputs=model.get_layer('simple_rnn').output)
intermediate_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp (InputLayer)    │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ inp[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ inp[0][0]         │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,200 (125.78 KB)

 Trainable params: 32,200 (125.78 KB)

 Non-trainable params: 0 (0.00 B)

Now, let's get the intermediate outputs for our input data `X`. Since the `SimpleRNN` layer in our model is configured with `return_sequences=False`, it outputs only the hidden state of the *last* time step. If `return_sequences=True` were used, it would output the hidden states for all time steps (pre-timestamps).

In [14]:
hidden_states = intermediate_model.predict(X)
print("Shape of hidden states (last time step):")
print(hidden_states.shape)
print("\nFirst 5 hidden states:")
print(hidden_states[:5])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Shape of hidden states (last time step):
(30, 8)

First 5 hidden states:
[[-0.2027226   0.1526764  -0.23071203  0.26159623 -0.24021335  0.06568942
   0.20396928 -0.22857282]
 [-0.68362993  0.5753635  -0.68135405  0.65958285 -0.7508098   0.20303304
  -0.04003427 -0.74018764]
 [-0.15088016 -0.00716204 -0.37654126  0.16355707 -0.30886447 -0.11570185
  -0.11997379 -0.29839283]
 [-0.39761737 -0.10474437 -0.7178274   0.5106014  -0.4776839  -0.17636313
  -0.335015   -0.56485367]
 [-0.45024645  0.6447259  -0.20972025  0.38448605 -0.6310682   0.40593013
   0.32305962 -0.3995527 ]]
